# displayDoc

In [23]:
# Display Docs (md files)
import os,sys
from pathlib import Path
DIR = Path.home() / 'GDrive/dev/playground/Howto'
if len([p for p in sys.path if 'HowTo' in p]) == 0 : 
    sys.path.append(str(DIR))
from mdDisplay import mdDisplay


outlineList = ['LLC_AccountingDesign.md',
               'LLC_AccountingWorkflow.md',
               'LLC_DataFlowDesign.md',
               'ProjectLevel3_v0.2 Book Activities.md',
               'ProjectLevel4_v0.3 Tax Activites.md',
               'Readme_aiCowork.md']

def docs(opt='All'):
    if opt == 'All':
        m = mdDisplay(DIR=outlineList)
        m.display()
        return

    # display 1 .md file
    m = mdDisplay(FN=opt)
    m.display()

#docs(opt ='IRS_Form1065_DataFlow.md')
m = mdDisplay(FN='LLC_DataFlowDesign.md')
c = m.mdMerge()
m.display()

# IRS Form 1065 — Book-to-IRS Data Flow

**Scope:** End-to-end pipeline that turns LLC book transactions into a filled
IRS Form 1065 PDF.  
**Phase:** ProjectLevel3 v0.2 (Book Activities) → ProjectLevel4 v0.3 (Tax Activities) boundary.  
**Last updated:** 2026-04-24 (v0.2.4.6)

---

## Why this doc exists

The per-module API reference for the IRS layer lives in
`irs/docs/irsForm1065_book2irsDeisng.md` and `irs/docs/Readme_Form1065.md`.
Those documents describe *what each class does*.  This doc instead answers a
different question:

> **How does a single rent check become a number in box `P1_1a` of the filed
> Form 1065 PDF?**

It follows the complete data path from the source JSON ledgers through the
immutable `stmt*` construction layer, into the IRS namespace/publish
machinery, and finally out to `Form1065_FILL.pdf`.  It is the canonical
overview for the v0.2 refactor that consolidated all data-wrangling into
`ledger/` and retired the ad-hoc `ui.llcIRSViewBase` loaders.

---

## LLC Data Flow High Level
- refer to Levels of Accounting in LLC_AccountingWorkflow.md.
  
!<a style='color:blue' href="llcDataFlowHL.svg">irsDataFlow</a>

## LLC Data Flow Levels 1-3
- refer to Levels of Accounting in LLC_AccountingWorkflow.md.
  
!<a style='color:blue' href="llcDataFlow_L1_3.svg">irsDataFlow</a>

## LLC Data Flow Levels 4-6
- refer to Levels of Accounting in LLC_AccountingWorkflow.md.
  
!<a style='color:blue' href="llcDataFlow_L4_6.svg">irsDataFlow</a>


---

## Layer responsibilities

| # | Layer | Module path | Responsibility | Mutates? |
|---|---|---|---|---|
| ① | **Source transactions** | `Accts/llc*_<LLC>.json` | Human-editable book-entry ledgers + entity profile + owner list | ✅ edited via the UI |
| ② | **Ledger** | `ledger/` | All data construction, wrangling, double-entry expansion, aggregation | ❌ constructed objects are immutable (`save()` raises `InvalidRequestError`) |
| ③ | **IRS** | `irs/` | Map ledger stmt cells → IRS fillDict entries; build and write FILL.pdf | ❌ fillDict write is idempotent — re-running rebuilds from stmt |
| ④ | **UI** | `ui/` + `ui/templates/` | Flask preview of the Form 1065 — thin pass-through, zero data wrangling | ❌ read-only view |
| ⑤ | **Filed artifacts** | `Forms_IRS/Form1065_*.{pdf,json}` | Cached fillDict + filed PDF — the deliverable | overwritten by `saveFILL()` |

---

## Book to IRS PDF (per `Readme_Form1065.md`)

```python
# Step 1 — discover AcroForm fields on the IRS template PDF
nspace   = form._buildNSpace()

# Step 2 — persist the namespace + a worksheet PDF
form.saveNSpace(nspace)

# Step 3 — resolve values + set publish flags (Publish / CPA:unknown / blank)
fillDict = form._buildFillDict(nspace)

# Step 4 — write FILL.pdf + fillDict.json cache
form.savePDF(fillDict)
```

`nSpaceMap()` (called during Step 3 and reused by the UI) returns
`Dict[(tblID, rowNm, colNm) → List[fillDict]]` — one book cell may fan out
into multiple PDF fields (e.g. a single `net_income` cell maps to Pg 1 line
23 **and** Sch K Pg 5 line 1 **and** Sch M-1 Pg 6 line 1).

---

## What changed in v0.2 (book vs. tax boundary)

| Concern | v0.1 (pre-refactor) | v0.2 (current) |
|---|---|---|
| Where is data wrangling? | Scattered across `ui/llcIRSViewBase.py` loaders, `stmtFinancialReport.taxData()`, and per-view helpers | **Only in `ledger/`** — stmt\* objects build themselves from source JSON, UI is a pure consumer |
| How does the view read `llc.entity` / `llc.F1065`? | Direct dict reads in `_llcIRSViewBase._ev` / `_fv_prof` | **`ledger.stmtProfile`** — row-addressable `stmtDB` subclass, same uniform API as every other `stmt*` |
| How is Form 1065 Pg 2-6 rendered? | Five separate Flask views (`llcForm1065SchBPg2`, `…Pg3`, `…Pg4`, `llcForm1065SchKPg5`, `llcForm1065Pg6`) | **One consolidated page** — `ui/llcForm1065.frames()` composes 6 collapsible frames, rendered by `form1065_view.html` |
| How are IS/BS values sourced for stats? | `stmtFinancialReport.taxData()` (requires `llc.bk` initialization) | Direct `stmtIncomeStmt` + `stmtBalanceSheet` queries (no `llc.bk` dependency) |
| Income-row sign convention | Depended on taxData JSON post-processing | Explicit: `stmtIncomeStmt` stores `Balance = Debit − Credit`, so Income rows have negative Balance; the view flips sign at display time |

---

## One transaction, end-to-end

Trace a single rent payment of **\$1,200** booked on 2025-06-15 all the way
through the pipeline:

1. **Book entry** — user adds a row to `llcExpRev_WBGroupLLC.json` via the
   Exp/Revenue editor: `{dt: "2025-06-15", amt: 1200.00, aType: "Credit",
   acct: "Rental_Income", acctType: "Income", desc: "June rent — unit 3"}`.
2. **GL expansion** — `ledger.stmtGeneralLedger` reads that JSON on
   instantiation, runs `toDoubleEntry()`, and emits **two** GL rows: one
   Credit against `Rental_Income`, one Debit against the corresponding cash
   account.  Rows are immutable after `_finalize()`.
3. **Aggregation** — `ledger.stmtTrialBalance` sums by `acctType`, so
   `Income.Rental_Income` rolls up alongside other revenue accounts.
4. **Income statement** — `ledger.stmtIncomeStmt` filters GL rows to
   `acctType ∈ {Income, Expense}`, giving the per-account Balance column.
5. **Form 1065 resolve** — `irs.Form1065._resolveTaxData()` pulls
   `stmtFinancialReport(llc).taxData()` (which composes from stmt\* under
   the hood), yielding `is_data['rent_income']` = 1200 (plus any other
   rent receipts).
6. **Namespace fan-out** — `Form1065.nSpaceMap()` binds
   `(tblID="Form1065", rowNm="L1a", colNm="Amount")` → the `P1_1a` PDF
   field entry.
7. **Publish decision** — `Form1065._buildFillDict()` sets
   `fillDict["f41"]["publish"] = True` (value is non-zero and sourced from
   ledger), with `value = "1,200.00"`.
8. **UI preview** — `ui.llcForm1065` asks `nSpaceMap()` for the same
   bindings, emits one UI row per fillDict, and the Flask template renders
   "1,200.00" in the Pg 1 frame under the Publish ViewBy filter.
9. **Filed output** — `Form1065.saveFILL()` writes
   `Form1065_FILL.pdf` (the filed PDF) and `Form1065_fillDict.json`
   (cache so subsequent views skip step 5).

---

## Invariants the diagram enforces

- **One-way flow.** Arrows never point backwards from right to left — nothing in `irs/` or `ui/` writes to `ledger/` or source JSON.  The only write is the final `saveFILL()` to `Forms_IRS/`.
- **stmt\* is the only data source for IRS.**  `irs/Form1065` never opens source JSON files directly; it goes through `stmtFinancialReport` (which itself only composes `stmt*`) or `stmtProfile` for header fields.
- **UI holds no data.**  `ui.llcForm1065` caches a `stmtProfile` for stats
  and calls `Form1065.nSpaceMap()` + sub-view `.load()` methods — it never
  merges, filters at aggregation level, or writes.
- **Every cell is addressable.**  Because every `stmt*` row carries
  `(tblID, rowNm, colNm)` and every fillDict entry carries `(fID, logicalKey, tblID, rowNm, colNm)`, you can trace any number on the
  filed PDF back to the exact book transaction that produced it.

---

## See also

- `irs/docs/irsForm1065_book2irsDeisng.md` — per-class API surface and the
  pre-refactor legacy architecture diagram (kept for historical reference;
  **not** modified in v0.2.4).
- `irs/docs/Readme_Form1065.md` — the 4-step IRS workflow in detail
  (`_buildNSpace` / `saveNSpace` / `_buildFillDict` / `saveFILL`).
- `docs/LLC_AccountingDesign.md` — book-level architecture (ledger → stmt).
- `docs/ProjectLevel4_v0.3 Tax Activites.md` — upcoming Phase 3 tax work
  that builds on this pipeline (K-1 generation, CPA review handoff).

------------


display(Markdown(![xxx](irsDataFlow.png)))